# AI Engineering Interview Prep
## Section: LLM Fundamentals

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 731+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (80 Qs)",
    "18. FastAPI (60 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 📚 Section 1 — LLM Fundamentals

### Q1. What are foundation models, and how have they changed AI engineering?
**A:** Foundation models are giant neural networks pre-trained on massive text (and image/code) corpora. They learn a general understanding of language and the world. Before them, every AI task needed its own training run. Now you pick Gemini or Llama, layer your application logic on top, and ship. That shift created the "AI Engineer" role — someone who uses, fine-tunes, and productionises these models rather than training from scratch.

🏷️ *All three of my projects (Nyaya-Pro, JobPilot AI, ExamGenie AI) use foundation models (Gemini, Llama) as the reasoning engine and build application logic on top — I've never trained from scratch.*


### Q2. What is a Large Language Model (LLM), and how does it work?
**A:** An LLM is a foundation model trained on text using next-token prediction on billions of sentences. It learns grammar, facts, reasoning patterns, and code. At inference: you send a prompt → it generates tokens one-by-one, each time predicting the most likely next token given everything before it. No magic — just very good next-token prediction at scale. Key caveat: LLMs have statistical associations, not true knowledge — which is why they hallucinate confidently.


### Q3. Inside ChatGPT: What happens after you hit Enter?
**A:**
1. **Tokenisation** — your text is split into sub-word tokens (e.g., "unhappiness" → ["un","happi","ness"])
2. **Embedding lookup** — each token maps to a dense vector
3. **Transformer forward pass** — 96+ layers of self-attention + FFN process all tokens in parallel
4. **KV cache** — previously computed Key/Value pairs are cached so only the new token needs full attention
5. **Logit projection** — the final hidden state is projected to vocabulary logits
6. **Sampling** — softmax + temperature + top-p/top-k → pick next token
7. **Repeat** steps 3-6 until EOS token or max length
8. **De-tokenise** → return text to user, usually streamed token-by-token


### Q4. What is the Transformer architecture and how does it work?
**A:** Introduced in "Attention is All You Need" (2017). Core idea: replace sequential RNNs with parallel **self-attention** — every token can attend to every other token simultaneously.

Building blocks (repeated N times):
- **Multi-Head Self-Attention** — contextualise each token w.r.t. all others
- **Add & Norm** (residual connection + LayerNorm) — training stability
- **Feed-Forward Network** (two linear layers with GELU) — token-level processing
- **Add & Norm** again

Plus: Embedding layer (input), Positional encoding, Linear + Softmax head (output).


### Q5. What are the key components of the Transformer architecture?
**A:**
| Component | Role |
|-----------|------|
| Token Embedding | Maps tokens → dense vectors |
| Positional Encoding | Adds position info (RoPE, sinusoidal, learned) |
| Multi-Head Self-Attention | Each token attends to all others via Q/K/V |
| Feed-Forward Network | Per-token non-linear transformation |
| Residual (Skip) Connections | Gradient highway for deep stacking |
| Layer Normalisation | Stabilises activations (RMSNorm in modern LLMs) |
| Output Linear + Softmax | Projects to vocabulary probabilities |


### Q6. What is tokenisation in LLMs?
**A:** Tokenisation converts raw text to integers the model can process. Text is split into sub-word pieces (tokens). Example: "ChatGPT" → ["Chat","G","PT"]. Rule of thumb: 1 token ≈ 4 chars ≈ ¾ of a word in English. Why sub-words? They handle rare words, typos and new terms gracefully — common words are single tokens, rare words are split into known pieces. Libraries: `tiktoken` (OpenAI), `sentencepiece` (Google).


### Q7. Explain BPE (Byte Pair Encoding).
**A:** BPE builds a vocabulary iteratively:
1. Start with individual characters as tokens
2. Count every adjacent pair in the corpus
3. Merge the most frequent pair into a new token
4. Repeat until vocabulary size is reached (e.g., 50,000)

Example: "l","o" appear together often → merge to "lo"; "lo","w" → "low"; eventually "lower" becomes one token.
BPE is used by GPT-2/3/4. Result: common words = single tokens, rare words = sub-word pieces seen during training.


### Q8. Explain WordPiece and SentencePiece.
**A:**
**WordPiece** (used by BERT): Similar to BPE but merges pairs that maximise the likelihood of the training data under a language model, not just raw frequency. Marks sub-words with `##` prefix: "playing" → ["play","##ing"].

**SentencePiece** (used by T5, Llama): Language-agnostic — treats the input as a raw byte/character stream without any pre-tokenisation (no whitespace splitting assumption). Works directly on raw Unicode. Great for multilingual models because it doesn't assume space-separated words (e.g., Chinese, Japanese). Uses BPE or unigram under the hood.


### Q9. What is positional encoding, and why is it needed in Transformers?
**A:** Self-attention is permutation-invariant — "cat bites dog" and "dog bites cat" look identical without positional info. Positional encoding injects position-dependent signals into each token's embedding.

- **Sinusoidal (original Transformer):** Fixed sine/cosine waves of different frequencies per position. No learnable parameters.
- **Learned embeddings (GPT-2):** A trainable vector per position.
- **RoPE (Rotary Position Embedding, used in Llama/Gemini):** Rotates Q and K vectors by position-dependent angles → relative positions encoded directly in attention scores. Generalises to longer sequences than seen during training.


### Q10. What are embeddings?
**A:** Embeddings are dense vector representations where semantic similarity = geometric proximity. "King" and "Queen" are close; "King" and "Pizza" are far.

Types:
- **Token embeddings** — lookup table mapping token ID → vector (learned during training)
- **Sentence/text embeddings** — single vector for a whole sentence/paragraph (from models like sentence-transformers)
- **Image/audio embeddings** — from vision/audio encoders for multi-modal use

🏷️ *In Nyaya-Pro, legal document chunks are embedded with FAISS so "bail conditions for murder" retrieves relevant BNS sections by semantic meaning, not just keyword match.*


### Q11. Explain Q (Query), K (Key), and V (Value) in attention.
**A:** Think library analogy:
- **Query** = what you're looking for ("books about criminal law")
- **Key** = the label on each book's spine
- **Value** = the actual content of the book

For each token: compute similarity of its Q against every other token's K → attention weights → weighted sum of all V's → contextualised output.

Mathematically:
```
Attention(Q,K,V) = softmax( QKᵀ / √d_k ) × V
```
The √d_k scaling prevents dot products from getting too large and pushing softmax into near-zero gradient regions.


### Q12. What is self-attention, and how does it work in Transformers?
**A:** Self-attention lets each token look at ALL other tokens to figure out context. For "The animal didn't cross the street because **it** was tired" — when processing "it", self-attention decides it should strongly attend to "animal".

Steps (for each token):
1. Project to Q, K, V via learned weight matrices
2. Compute Q·Kᵀ → similarity scores
3. Scale by √d_k, apply softmax → attention weights (sum to 1)
4. Weighted sum of V → output

Runs in parallel for all tokens — hence much faster than RNNs on GPUs.


### Q13. What is Cross Attention in Transformers?
**A:** Cross-attention is like self-attention except **Q comes from one sequence and K,V come from another**.

Used in encoder-decoder models (T5, original Transformer):
- Encoder processes source → produces K, V
- Decoder generates target — its Q attends to encoder's K,V

In RAG systems: the LLM's Q attends to retrieved document tokens (K,V) — this is how the model "reads" the context you inject. Cross-attention is also how vision-language models let text attend to image patch tokens.


### Q14. Why do we scale dot product attention by √d_k?
**A:** When d_k (dimension of Q and K) is large (e.g., 128), random dot products Q·K have variance proportional to d_k. Large magnitudes push softmax output to near-zero or near-one values — an almost one-hot distribution — causing **vanishing gradients** during training.

Dividing by √d_k normalises the variance to 1 regardless of d_k, keeping softmax in a reasonable range where gradients flow well.

Analogy: If you're scoring students on a 100-point scale vs a 10,000-point scale, you'd normalise before comparing.


### Q15. What is causal masking?
**A:** Decoder-only LLMs (GPT, Llama) generate left-to-right. During training, the model processes all tokens at once (parallel) but must NOT see future tokens — that would be cheating.

Causal masking sets all upper-triangle attention weights to -∞ before softmax, which becomes 0 after softmax. Token at position i can only attend to positions ≤ i.

Result: the model learns to predict token_i using only tokens 0..i-1 — exactly what inference requires. Without this, the model would "cheat" by looking ahead and fail completely at generation.


### Q16. What are multi-head attention mechanisms? Why use multiple heads?
**A:** Instead of one attention computation, multi-head attention runs H parallel attention heads with different Q,K,V weight matrices. Each head learns to attend to different aspects:
- Head 1: grammatical subjects
- Head 2: coreferences ("it" → "cat")
- Head 3: semantic roles
- etc.

Outputs of all H heads are concatenated and projected back:
```
MultiHead(Q,K,V) = Concat(head_1,...,head_H) × W_O
```
More heads = richer contextual representations. GPT-3 uses 96 heads. Using just 1 head, the model could only learn one relationship pattern per layer.


### Q17. What are Feed-Forward Networks (FFN) in LLMs?
**A:** After self-attention, each token's representation passes independently through a small 2-layer MLP:
```
FFN(x) = GELU(x × W1 + b1) × W2 + b2
```
Expansion ratio is typically 4× (768-dim model → 3072-dim hidden). Modern LLMs use SwiGLU gating.

Key insight from research: **Attention captures relationships between tokens; FFN stores factual knowledge** ("Paris is the capital of France" is in the FFN weights). FFN parameters are ~⅔ of total model parameters.


### Q18. What is the context window in LLMs, and why does it matter?
**A:** The context window is the max tokens the model can process at once — prompt + response combined.

It matters because:
- Documents longer than context must be chunked (design implication for RAG)
- Attention is O(N²) in sequence length — 2× context = 4× compute
- "Lost in the middle": models recall beginning and end better than middle of long contexts
- Cost: longer context = more tokens billed

Current sizes: GPT-4o = 128K, Claude 3.5 = 200K, Gemini 1.5 Pro = 1M.

🏷️ *Nyaya-Pro's full legal corpus is 140K+ words. I chunk + retrieve only top-5 relevant sections (~2K tokens) to stay well within Gemini's context window.*


### Q19. Why is the context window limited in LLMs?
**A:** Three fundamental reasons:

1. **Quadratic attention complexity** — standard attention is O(N²) in memory and compute. Doubling context → 4× resources. At 1M tokens, this is impractical without tricks (FlashAttention, sparse attention).

2. **KV cache memory** — every token requires Key and Value vectors stored for each layer. For a 70B model with 32 layers, a 100K token context uses ~50GB just for KV cache.

3. **Training data scarcity** — very long-range dependencies are rare in training data, so the model may not generalise well beyond its trained context length, even if the infrastructure could handle it.


### Q20. What is temperature in the context of LLMs, and how does it affect output?
**A:** Temperature T scales logits before softmax: `P(token_i) = exp(z_i / T) / Σ exp(z_j / T)`

| Temperature | Effect | Use case |
|-------------|--------|----------|
| 0 | Always picks highest-probability token (deterministic) | Code, facts, structured output |
| 0.3–0.7 | Balanced — predictable but not robotic | Q&A, summarisation |
| 0.7–1.0 | Creative, varied | Chatbots, writing |
| >1.5 | Very random, often incoherent | Brainstorming only |

Rule: lower T = higher precision, lower recall. Higher T = more diversity but more hallucination.


### Q21. Why is the first token slower than the rest in an LLM?
**A:** This is the **Time to First Token (TTFT)** problem.

For the **first token**: the model must run attention over the **entire prompt** — O(N²) computation for all input tokens. This is the "prefill" phase. Expensive.

For **subsequent tokens**: each new token only needs attention with existing tokens, but crucially, the KV cache stores all previously computed Key/Value pairs — so you only compute new Q for the new token and attend to cached K,V. This "decode" phase is much faster.

Result: streaming starts slow then accelerates. TTFT is a key latency metric for production LLM serving.


### Q22. Explain Top-p (nucleus) sampling and Top-k sampling. How do they differ?
**A:**
**Top-k**: Consider only the k most probable tokens. If k=50, sample from top-50 tokens only. Fixed filter regardless of distribution shape.

**Top-p (nucleus)**: Consider the smallest set of tokens whose cumulative probability ≥ p. If p=0.9 and the top-3 tokens already sum to 90%, use only those 3. If the distribution is flat, you might need 500 tokens to reach 90%.

**Difference**: Top-k is fixed-size; top-p is adaptive. Top-p is generally preferred because it's more conservative when the model is confident (few options) and more diverse when uncertain (many options). Most LLMs use top-p = 0.9, temperature = 0.7 as defaults.


### Q23. What are logits, and how are they used in text generation?
**A:** Logits are the raw, unnormalised scores output by the LLM's final linear layer — one number per vocabulary token (e.g., 128,256 numbers for Llama 3).

Pipeline:
1. Final hidden state → Linear(W_out) → logits vector [vocab_size]
2. Apply temperature: logits / T
3. Apply top-k/top-p filtering (set non-candidates to -∞)
4. Softmax → probability distribution
5. Sample token from distribution
6. Decode token ID → text, append to sequence, repeat

High logit = model thinks this token is likely next. Logits can be negative. You can also use logits directly for classification tasks (argmax gives the predicted class).


### Q24. What are skip connections (residual connections) in Transformers?
**A:** Skip connections add the input of a sub-layer to its output:
```
output = LayerNorm(x + SubLayer(x))
```
Why critical? Without them, gradients vanish when backpropagating through dozens of layers — early layers get no learning signal. Skip connections provide a "gradient highway" that bypasses each layer.

Secondary benefit: layers learn **residual corrections** to the identity function rather than full transformations — much easier to optimise. GPT-3 has 96 layers; without residuals, training would be impossible.


### Q25. What is the difference between open-source and closed-source LLMs?
**A:**
| Factor | Open-source (Llama, Mistral, Gemma) | Closed-source (GPT-4, Claude, Gemini) |
|--------|-------------------------------------|---------------------------------------|
| Cost | Free weights, pay compute | Pay per token |
| Data privacy | Stays on your infra | Sent to provider |
| Customisation | Full — fine-tune, quantize | Limited |
| Quality | Catching up fast | Frontier tasks: still ahead |
| Setup | Needs infra expertise | API call — done |
| Compliance | Easier (data stays local) | Depends on provider's DPA |

Choose open-source: data privacy requirements, very high volume, need fine-tuning control.
Choose closed-source: need best quality now, small team, low engineering overhead.

🏷️ *JobPilot AI: Ollama (local Llama) for dev, Groq API (Llama-3.3-70b) in production — best of both worlds.*


### Q26. What is the difference between encoder-only, decoder-only, and encoder-decoder architectures?
**A:**
**Encoder-only (BERT, RoBERTa):**
- Bidirectional attention — each token sees all others
- Great for understanding: classification, NER, embeddings
- Can't generate text

**Decoder-only (GPT, Llama, Gemini):**
- Causal (left-to-right) attention
- Generates text autoregressively
- All modern LLMs — can do understanding via prompting

**Encoder-Decoder (T5, BART, original Transformer):**
- Encoder reads input (bidirectional), Decoder generates output (causal)
- Natural for seq2seq: translation, summarisation
- More complex to serve; less popular now

Modern trend: decoder-only wins — simpler, scales better, flexible via prompting.


### Q27. What is KV cache, and how does it speed up inference?
**A:** During generation, computing attention for each new token requires the Key and Value vectors of ALL previous tokens. Without caching, you'd recompute them every step — O(N²) waste.

**KV cache** stores K and V for all past tokens in memory. For each new token:
1. Compute only the new token's Q
2. Concatenate with cached K,V
3. Compute attention → output

Memory cost: for a 7B model, KV cache at 4K tokens ≈ 4GB. At 100K tokens with 100 concurrent users → hundreds of GBs. This is why Paged Attention (vLLM) and KV cache management matter enormously in production.

🏷️ *Nyaya-Pro uses Gemini's prompt caching API to cache the system prompt + Constitution context across requests — saves ~75% token cost for returning users.*


### Q28. What is model distillation, and how is it used with LLMs?
**A:** Knowledge distillation trains a smaller "student" model to mimic a larger "teacher" model. Instead of just training on hard labels (0 or 1), the student learns from the teacher's **soft probability distributions** — which carry richer information (e.g., the teacher thinks "cat" is 60% likely, "kitten" is 35%, "dog" is 5% — vs just label "cat").

Steps:
1. Run teacher on training data → collect output distributions
2. Train student using KL-divergence loss between its outputs and teacher's distributions

Examples: GPT-4o mini (distilled from GPT-4o), Llama 3 8B can be distilled from 70B. Used to make smaller, deployable models that approximate much larger ones.


### Q29. What is Mixture of Experts (MoE), and how does it work?
**A:** Instead of activating all parameters for every token, MoE has multiple parallel "expert" FFN networks plus a **router** that selects which 1-2 experts handle each token.

```
token → Router (softmax over experts) → top-2 experts → weighted sum of outputs
```

Example: Mixtral 8×7B has 8 experts (46.7B total params), activates 2 per token (12.9B active). You get large-model quality at small-model inference cost.

Trade-offs: More memory to load all experts; complex distributed serving; but better throughput per FLOP. Newer models (Llama 4, DeepSeek-V3) heavily use MoE.


### Q30. What is the difference between dense and sparse models?
**A:**
**Dense models (GPT-4, Llama):** Every parameter is used for every token during every forward pass. Straightforward but compute-intensive.

**Sparse models (MoE architectures):** Only a fraction of parameters are active per token (controlled by the router). The rest sit idle.

Dense: predictable, simple to optimise, every param contributes.
Sparse: computationally efficient, but needs all params loaded in memory, routing overhead, potential for load-imbalance (some experts underused).

Modern trend: sparse MoE at scale (DeepSeek-V3 uses 671B total / 37B active per token).


### Q31. What is Flash Attention?
**A:** Standard attention loads the full N×N attention matrix to GPU HBM (slow memory) and reads/writes it multiple times during softmax + dropout — the bottleneck for long sequences.

Flash Attention (Tri Dao, 2022) is an **I/O-aware** algorithm:
- Splits Q,K,V into tiles that fit in SRAM (fast GPU on-chip memory)
- Fuses the attention computation into one kernel — no HBM reads for the intermediate N×N matrix
- Uses online softmax to handle tiling correctly

Result: **same output as standard attention** but 2-4× faster and uses O(N) memory instead of O(N²). Now standard in every major framework (PyTorch, vLLM, HuggingFace). Flash Attention 3 on H100 achieves ~75% of theoretical peak FLOP/s.


### Q32. What is Cross-Entropy Loss?
**A:** Cross-entropy loss measures how well the model's predicted probability distribution matches the true label. For LLM next-token prediction:
```
Loss = -log P(true_next_token | context)
```
If the model assigns 90% probability to the correct next token → loss = -log(0.9) ≈ 0.1 (low, good).  
If it assigns 1% → loss = -log(0.01) ≈ 4.6 (high, bad).

The LLM is trained to minimise the average cross-entropy loss over all tokens in the training corpus. **Perplexity = exp(cross-entropy loss)** — a common LLM quality metric. Lower perplexity = better language model.


### Q33. What is Grouped-Query Attention (GQA)?
**A:** In Multi-Head Attention (MHA), every head has its own Q, K, V — expensive KV cache.
In Multi-Query Attention (MQA), all heads share ONE K,V — small cache but quality drops.
GQA is the middle ground: heads are divided into groups; each group shares K and V.

```
MHA: H heads × H KV pairs
MQA: H heads × 1 KV pair
GQA: H heads × G KV pairs (G < H)
```

Llama 3 70B uses GQA with 8 KV heads for 64 query heads. Reduces KV cache size 8× with negligible quality loss. Critical for serving large models with long contexts efficiently.


### Q34. How does Rotary Position Embedding (RoPE) work?
**A:** RoPE encodes position by **rotating** the Q and K vectors in 2D sub-spaces using position-dependent rotation matrices:
```
Q_rotated(pos) = Q × R(pos)
K_rotated(pos) = K × R(pos)
```
When you compute Q·K = Q_rotated(m)·K_rotated(n), the result depends only on the **relative position (m-n)**, not absolute positions. This is automatically built into the attention score without extra parameters.

Why preferred: (1) Naturally encodes relative positions, (2) Extrapolates to longer sequences via RoPE scaling tricks (YaRN, LongRoPE), (3) No extra parameters to learn. Used in Llama 2/3, Mistral, Gemini, Qwen.


### Q35. Explain Layer Normalisation.
**A:** Layer Norm normalises activations **across the feature dimension** (not batch):
```
LN(x) = (x - μ) / σ × γ + β   where μ,σ computed over features of one token
```
Keeps activations from exploding or vanishing across layers — critical for deep Transformer training. Applied before each sub-layer in pre-norm architectures (modern LLMs).

**vs Batch Norm:** Batch Norm normalises across the batch dimension (requires large batches, bad for sequence models with variable length). Layer Norm normalises per sample per token — works with batch size = 1.


### Q36. Explain RMSNorm (Root Mean Square Layer Normalization).
**A:** RMSNorm is a simplified LayerNorm that skips computing the mean (re-centring) and only normalises by the RMS:
```
RMSNorm(x) = x / RMS(x) × γ    where RMS(x) = sqrt(mean(x²) + ε)
```
Why? Empirically, the re-centring (subtracting mean) in LayerNorm contributes little to training stability, but the re-scaling (dividing by std) does most of the work. RMSNorm is ~10% faster with equivalent quality.

Used in: Llama 1/2/3, Mistral, Gemma, Qwen — essentially all modern open-source LLMs.


### Q37. Your LLM keeps ignoring instructions. How do you make it follow structured output formats?
**A:** In order of reliability:
1. **Use structured output APIs** — OpenAI (`response_format={"type":"json_object"}`), Gemini (`response_mime_type="application/json"`), Groq — forces valid JSON at the token level
2. **Pydantic + Instructor library** — wrap LLM call with schema validation + auto-retry
3. **Clear system prompt** — "Respond ONLY with valid JSON matching this schema exactly: {schema}. No prose, no markdown."
4. **Few-shot examples** — show 2-3 input→output JSON pairs
5. **Grammar-constrained decoding** — llama.cpp supports formal grammar rules at token generation level

🏷️ *In Nyaya-Pro, every LLM call uses a Pydantic model for the response (answer, cited_sections, confidence). If parsing fails, a retry decorator retries up to 3 times with an error message appended to the prompt.*


### Q38. Your LLM hits the context window limit on long documents. How do you handle it?
**A:**
1. **RAG (recommended)** — chunk the document, embed, retrieve only the relevant pieces. Never stuffing the whole doc.
2. **Map-reduce summarisation** — process each chunk independently, then summarise the summaries
3. **Sliding window** — process overlapping windows, aggregate answers
4. **Hierarchical indexing** — build a tree of summaries at multiple granularities
5. **Upgrade the model** — Gemini 1.5 Pro (1M tokens) handles very long docs natively
6. **Context compression** — extract only the key sentences relevant to the query before sending

🏷️ *Nyaya-Pro: Indian Constitution alone = 140K+ words. I chunk all legal texts at 512 tokens, embed in FAISS, retrieve top-5 relevant chunks (~2,500 tokens) per query — stays well within limits.*


### Q39. Your LLM does not admit when it doesn't know. How do you make it say "I don't know"?
**A:**
1. **Explicit system prompt instruction:** "If you are not confident or the answer is not in the provided context, say 'I don't have enough information to answer this confidently' — never guess."
2. **RAG with strict grounding:** "Answer ONLY using the context provided. If it's not there, say so."
3. **Calibration prompt:** After generating an answer, ask "Rate your confidence 1-10. If below 7, revise to say you're unsure."
4. **Output classifier:** Run a separate LLM call: "Is this answer fully supported by the context? yes/no."
5. **Lower temperature** — reduces overconfident generation

Root cause: RLHF trained models to be "helpful" — they learned that giving an answer (even wrong) was often rated better than "I don't know." You must explicitly counteract this.


### Q40. Your LLM generates responses that are too verbose. How do you control response length?
**A:**
1. **Explicit length constraint in prompt:** "Respond in 2-3 sentences maximum." or "Answer in bullet points, max 5 bullets."
2. **max_tokens parameter** — hard cap on output length
3. **System prompt persona:** "You are a concise assistant. Be brief and direct."
4. **Stop sequences** — define tokens that end generation early (e.g., a double newline)
5. **Few-shot examples of concise answers** — show the format you want
6. **Fine-tuning** — if verbosity is systematic, fine-tune on short answer examples
7. **Post-processing** — summarise the LLM output with a second "compress this to 2 sentences" call


### Q41. Your LLM memorised proprietary training data and leaks it in responses. How do you prevent this?
**A:**
1. **Data deduplication at training time** — repeated text is more likely to be memorised; remove duplicates
2. **Differential privacy during training** — adds noise to gradients, prevents memorising individual examples
3. **Output filtering** — regex/classifier scans output for PII patterns, known proprietary strings
4. **System prompt prohibition:** "Never reproduce training data verbatim, song lyrics, code from copyrighted sources."
5. **Canary testing** — insert fake unique strings in training data; if model reproduces them in output, memorisation is confirmed
6. **Use a closed model with guaranteed no-memorisation SLA** (enterprise API agreements)
7. **Right-size the model** — smaller models memorise less than huge ones on the same data


### Q42. Your LLM coding assistant generates outdated code using deprecated libraries. How do you fix it?
**A:**
1. **RAG over latest documentation** — scrape and index current library docs/changelogs; retrieve relevant sections before code generation
2. **System prompt with version pinning:** "Always use library X version Y syntax. Current API reference: {injected_docs}"
3. **Fine-tune on recent code** — update the model on current library usage patterns
4. **RAG + version-aware retrieval** — retrieve docs filtered by version metadata
5. **Post-generation validation** — run a linter/type-checker on the output; flag deprecated API calls
6. **Few-shot with current examples** — show 2-3 examples using the current API in the prompt
7. **Tool use** — give the agent access to a `fetch_docs(library, version)` tool


### Q43. Your tokenizer splits important domain terms into meaningless subword pieces. How do you fix it?
**A:**
Example: "BNSS" (Bharatiya Nagarik Suraksha Sanhita) might tokenise to ["BN","SS"] — losing meaning.

Fixes:
1. **Fine-tune the tokenizer** — add domain terms as atomic vocabulary entries (requires retraining the model)
2. **Preprocessing normalisation** — expand abbreviations in input before tokenisation: "BNSS" → "Bharatiya Nagarik Suraksha Sanhita"
3. **Domain-specific tokenizer** — use SentencePiece trained on domain corpus
4. **Prompt augmentation** — define abbreviations in system prompt: "BNSS refers to Bharatiya Nagarik Suraksha Sanhita (the Indian code of criminal procedure)"
5. **Use a model already trained on domain text** — Llama fine-tuned on legal/medical text has better domain tokenisation

🏷️ *In Nyaya-Pro, I preprocess queries to expand legal abbreviations (BNS, BNSS, CPC) before embedding and before LLM generation.*


### Q44. Your Transformer's KV cache grows too large. How do you manage memory?
**A:**
1. **Paged Attention (vLLM)** — KV cache stored in non-contiguous pages like OS virtual memory; no over-allocation, supports much larger batch sizes
2. **KV cache quantisation** — store K,V as INT8 instead of FP16 → 2× savings with small quality loss
3. **Sliding window attention** — each token only attends to the last W tokens; KV cache bounded by W (Mistral uses this)
4. **GQA / MQA** — fewer K,V heads means proportionally smaller cache
5. **KV cache offloading** — move cold cache entries from GPU to CPU memory (slower access but frees GPU)
6. **Reduce max_sequence_length** — cap generation length
7. **Increase batch through lower per-request cache** — use smaller models for cache-intensive workloads


### Q45. Your Transformer runs out of memory on long documents due to quadratic self-attention. How do you scale it?
**A:**
1. **Flash Attention** — same result as standard attention but O(N) memory instead of O(N²) via tiled SRAM computation
2. **Sparse attention** — attend only to local windows + a few global tokens (Longformer, BigBird)
3. **Linear attention approximations** — approximate softmax attention in O(N) (Performer, Hyena)
4. **Sliding window attention** — Mistral's approach: each token attends to W=4096 previous tokens max
5. **Chunked attention** — process attention in manageable chunks, accumulate results
6. **RAG instead of long context** — don't send 100K tokens; retrieve 2K of the most relevant
7. **Gradient checkpointing** (training) — recompute activations during backward pass instead of storing them


### Q46. Your distilled student model fails on complex reasoning that the teacher handled. How do you close the gap?
**A:**
1. **Chain-of-thought distillation** — distill the teacher's reasoning steps, not just final answers. Include the CoT in training targets.
2. **Increase student capacity** — use a larger student model
3. **Better distillation objectives** — use intermediate layer matching (feature-level distillation), not just output KL divergence
4. **Data augmentation with hard examples** — generate more examples of the specific reasoning patterns the student fails on
5. **Curriculum learning** — start distillation on simple examples, gradually increase complexity
6. **Selective distillation** — only distill on tasks where the student fails; don't dilute with easy tasks
7. **Accept the gap** — some capabilities only emerge at scale; choose a task scope the student model can handle


### Q47. After RLHF, your LLM became safer but lost capability on hard tasks. How do you manage the alignment tax?
**A:** The "alignment tax" is the performance degradation on complex tasks after safety fine-tuning.

Mitigations:
1. **Better RLHF data** — collect preference data that rewards both safety AND capability; don't just reward refusal
2. **DPO instead of PPO** — DPO can achieve alignment with less capability degradation
3. **RLHF only on safety-critical domains** — don't align the full model; use routing to apply safety filters only where needed
4. **Separate capability and safety** — have a capable base model + a lightweight safety wrapper/filter
5. **Increased SFT data quality** — better instruction-following data before RLHF reduces how much RLHF needs to compensate
6. **Evaluate systematically** — track performance on hard benchmarks (MMLU, HumanEval) throughout RLHF; stop early if capability drops sharply


### Q48. Your RLHF-trained LLM is gaming the reward model. How do you fix reward hacking?
**A:** Reward hacking = the LLM finds patterns that score high on the reward model without being genuinely good (e.g., verbose, flattering, safely useless answers).

Fixes:
1. **Reward model ensembling** — train multiple reward models, use the minimum score (hard to fool all simultaneously)
2. **KL penalty** — penalise divergence from the original SFT model to prevent extreme optimisation
3. **Diverse preference data** — use raters from different demographics, backgrounds
4. **Constitutional AI (Claude approach)** — add explicit principle-based constraints alongside the reward signal
5. **Periodic reward model updates** — retrain the reward model on newly generated outputs to close the gap
6. **RLAIF** — use AI-generated critique (Constitutional AI) instead of/alongside human preference to scale diverse feedback


### Q49. Your chatbot loses context after 10 turns. How do you maintain long conversation context?
**A:**
1. **Sliding window** — keep only the last N turns (e.g., last 5) in the context window
2. **Summarisation memory** — periodically summarise older turns into a compact summary, keep summary + recent turns
3. **External memory (vector DB)** — store past turns as embeddings; retrieve relevant past messages when needed
4. **Summary buffer** — LangChain's ConversationSummaryBufferMemory: keeps recent turns verbatim + running summary of older ones
5. **Session state in DB** — persist conversation state server-side, reconstruct context on each request
6. **Use a longer context model** — Gemini 1.5 (1M tokens) can hold very long conversations natively

🏷️ *Nyaya-Pro: I compress conversation history after 5 turns into a brief summary ("Earlier: user asked about bail conditions under BNS"), then keep last 2 turns verbatim. Stored per user session in PostgreSQL.*


### Q50. Your chatbot fails when users switch topics mid-conversation. How do you handle topic switches?
**A:**
1. **Intent detection on each turn** — classify each message to detect topic shift; reset relevant context but keep user metadata
2. **Per-topic context isolation** — maintain separate context "slots" for different topics; switch between them on topic change
3. **Explicit acknowledgement prompt** — instruct the LLM: "If the user shifts to a new topic, explicitly acknowledge the switch and handle independently."
4. **State machine design** — model conversation as states; transitions between states on topic change
5. **Clear conversation history** — on detected topic switch, summarise previous topic and start fresh context for new topic
6. **Structured conversation graph** — use LangGraph with separate nodes per topic domain; route dynamically


### Q51. Your QA system always generates an answer even when no answer exists in context. How do you detect unanswerable questions?
**A:**
1. **Strict system prompt:** "If the answer is not explicitly present in the provided context, output exactly: 'UNANSWERABLE' — do not guess."
2. **Extractive QA approach** — force the model to point to the exact span in the context; if no span fits, it must say unanswerable
3. **Faithfulness score threshold** — compute faithfulness (RAGAS or NLI model) between answer and context; if score < 0.5, flag as hallucinated/unanswerable
4. **Two-step pipeline** — Step 1: "Does the context contain the answer? yes/no." Step 2: Only generate answer if yes.
5. **Fine-tune on SQuAD 2.0** — a dataset with unanswerable questions; teaches the model to distinguish
6. **Retrieval confidence check** — if retrieval similarity score < threshold, skip generation entirely


### Q52. Your summarisation system hallucinated facts not in the original article. How do you fix it?
**A:**
1. **Faithfulness-first prompting:** "Summarise ONLY the information present in the article. Do not add any facts, examples, or details not explicitly stated."
2. **Extractive → abstractive pipeline** — first extract key sentences (no hallucination risk), then rephrase those sentences only
3. **Citation enforcement** — require each summary sentence to include a quote from the source
4. **NLI-based faithfulness checker** — use a Natural Language Inference model to verify each summary claim is entailed by the source
5. **Temperature = 0** — deterministic output is less likely to hallucinate
6. **Fact verification loop** — run a second LLM call to check "Is this fact present in the article?" for each claim


### Q53. Your text generation repeats phrases in long outputs. How do you fix repetition?
**A:**
1. **Repetition penalty** — most LLM APIs support `repetition_penalty` (e.g., 1.1-1.3); penalises tokens that have appeared recently
2. **`no_repeat_ngram_size`** — prevents any n-gram from appearing twice (HuggingFace parameter)
3. **Higher temperature** — slightly more randomness breaks repetitive loops
4. **Presence/frequency penalty** (OpenAI API) — `presence_penalty` discourages any previously seen token; `frequency_penalty` penalises proportional to how often a token appeared
5. **Better prompting** — "Avoid repeating the same ideas. Cover each point once and move on."
6. **Truncate before looping** — detect repetition in post-processing (e.g., repeated 4-gram) and cut output there


### Q54. Transformers work on text — can they also understand images?
**A:** Yes! Vision Transformers (ViT) apply the same architecture to images:
1. **Patch embedding** — split image into 16×16 pixel patches
2. **Linear projection** — each patch → a vector (same as a token)
3. **Positional encoding** — add 2D positional info
4. **Standard Transformer encoder** — patches attend to each other

For vision-language models (GPT-4V, LLaVA, Gemini): a vision encoder (ViT or CNN) converts the image to patch vectors, these are projected to the LLM's embedding dimension and concatenated with text tokens — the LLM then processes both together.

🏷️ *ExamGenie AI sends PDF pages as images to Gemini 2.5 Flash — it processes both text content and diagrams/formulas simultaneously to generate contextually correct MCQs.*


### Q55. Small Language Models (SLMs)
**A:** SLMs are compact LLMs (1B-7B parameters) optimised for specific tasks or deployment constraints.

Examples: Microsoft Phi-3 (3.8B), Google Gemma 2B, Llama 3 8B, Apple OpenELM.

Advantages: Fast inference (CPU-capable), low cost, on-device capable, fine-tunable on consumer hardware.
Disadvantages: Weaker general reasoning, smaller context, less knowledge breadth.

Best for: Mobile/edge AI, high-volume specialised tasks, privacy-sensitive local deployment, cost-sensitive production where quality bar is achievable.

The key insight: a fine-tuned 7B SLM on a specific task can outperform a prompted 70B model on that task — specialisation beats size for narrow domains.


### Q56. Large Reasoning Models (LRMs)
**A:** LRMs (OpenAI o1/o3, DeepSeek-R1) are trained with extended chain-of-thought reasoning using RL to improve multi-step reasoning.

How they differ: Before answering, they generate internal "thinking" tokens (a scratchpad). They self-verify, backtrack and explore multiple reasoning paths before committing to an answer. This is "test-time compute scaling" — more compute at inference = better answers.

Training: GRPO (DeepSeek-R1) or process reward models that score reasoning steps, not just final answers.

Best for: Math, coding, logic puzzles, complex analysis. Not for: Simple Q&A (overkill, slow, expensive). A key shift: in LLMs, train more. In LRMs, think more.


### Q57. What are Autoregressive Models?
**A:** Autoregressive models generate sequences by predicting one element at a time, conditioned on all previous elements:
```
P(x₁,...,xₙ) = P(x₁) × P(x₂|x₁) × P(x₃|x₁,x₂) × ... × P(xₙ|x₁,...,xₙ₋₁)
```
All decoder-only LLMs are autoregressive. Advantages: flexible, can generate arbitrarily long sequences, principled probabilistic framework.

In images: PixelCNN generates pixels left-to-right, top-to-bottom. In audio: WaveNet generates audio samples one at a time. The paradigm is universal — any sequence generation problem can be framed as next-element prediction.


### Q58. Explain the difference between autoregressive and masked language modeling.
**A:**
**Autoregressive (GPT-style):** Predict next token given all previous tokens. Unidirectional (left-to-right). Training objective: maximise P(xₜ | x₁,...,xₜ₋₁) for all t. Great for generation.

**Masked Language Modeling (BERT-style):** Randomly mask 15% of tokens; predict them given ALL other tokens (bidirectional). Training objective: reconstruct masked tokens. Great for understanding (classification, NER) — but can't generate text coherently because it doesn't know what comes before AND after in a generation context.

Modern trend: Autoregressive wins because decoder-only models can do understanding too (via prompting/fine-tuning), while MLM models fundamentally can't generate.


### Q59. Proximal Policy Optimization (PPO)
**A:** PPO is the RL algorithm used in RLHF to fine-tune LLMs based on reward model scores.

Core idea: Update the policy (LLM) to maximise reward while not straying too far from the reference policy (original SFT model). The "proximal" part = clip the policy update ratio to [1-ε, 1+ε] so updates are stable.

```
L_PPO = E[min(rₜ × Aₜ,  clip(rₜ, 1-ε, 1+ε) × Aₜ)]
```
where rₜ = new_policy / old_policy ratio, Aₜ = advantage (how much better than baseline).

Why not vanilla RL? LLM action space is the full vocabulary at each step — huge. PPO's clipping prevents catastrophic divergence. Still complex — requires 4 models simultaneously: actor, critic, reward model, reference model.


### Q60. Direct Preference Optimization (DPO)
**A:** DPO aligns LLMs with human preferences without needing a separate reward model or RL training loop.

Given pairs of (preferred, rejected) responses, DPO directly optimises:
```
L_DPO = -E[log σ(β × log(π_θ(y_w)/π_ref(y_w)) - β × log(π_θ(y_l)/π_ref(y_l)))]
```
Where y_w = preferred, y_l = rejected, π_ref = reference model.

Simpler than RLHF:
- No reward model to train
- No PPO instability
- One training phase instead of three
- Similar or better alignment quality

Used by: Llama 3 Instruct, Mistral Instruct, Zephyr — essentially all modern open-source aligned models.


### Q61. Group Relative Policy Optimization (GRPO)
**A:** GRPO (used in DeepSeek-R1) is a PPO variant that eliminates the critic/value model:

Instead of a separate critic estimating the value baseline:
1. Generate G outputs for the same prompt (a "group")
2. Compute reward for each output
3. Use the group's **mean reward** as the baseline
4. Update the policy based on each output's advantage = (reward - group_mean_reward)

Benefits:
- No critic model to train and maintain (halves model count vs PPO)
- Works well for tasks where you can easily generate and evaluate multiple solutions (math, code)
- More sample-efficient for reasoning tasks

This is what made DeepSeek-R1 computationally efficient to train.


### Q62. Recursive Language Models (RLMs)
**A:** RLMs are a conceptual extension where a language model can call itself recursively to solve problems. Instead of a single forward pass, the model breaks a complex problem into sub-problems and recursively calls a language model to solve each sub-problem.

This is closely related to: divide-and-conquer prompting, hierarchical CoT, agentic self-reflection. In practice, this manifests as agents that spawn sub-agents for sub-tasks.

Example: "Write a research report" → recursively call LLM for each section → recursively call for citations → synthesise.

The theoretical appeal: RLMs can solve problems of arbitrary complexity by decomposition, rather than being bounded by context window or reasoning depth in a single pass.


### Q63. Continual Learning in LLMs
**A:** Continual learning = ability to learn new tasks/knowledge over time without forgetting old ones (avoiding catastrophic forgetting).

Challenges for LLMs:
- Fine-tuning on new data overwrites old knowledge
- Full retraining is prohibitively expensive

Approaches:
1. **Replay** — mix old training data into new fine-tuning batches
2. **LoRA adapters** — train new adapters for new domains without touching base weights
3. **Elastic Weight Consolidation (EWC)** — regularise against changing important weights
4. **Progressive Neural Networks** — new task gets new sub-network, old ones frozen
5. **RAG as a complement** — don't try to bake new facts into weights; use RAG for dynamic knowledge

Practical recommendation: For LLMs, use RAG for knowledge updates and LoRA for behaviour changes. True continual learning at LLM scale remains an open research problem.
